# Incertidumbre de un comité de modelos MACE

**Idea.** Entrenamos 5 modelos MACE con semillas distintas (el *comité*). Donde los 5 coinciden, la predicción
es fiable; donde discrepan, no. Este notebook mide esa discrepancia, la convierte en una incertidumbre en
meV/Å y comprueba, frente a DFT, si de verdad predice el error en varios conjuntos de configuraciones.

**Método**, en cuatro pasos:
1. **Predicción** = media de los 5 modelos.
2. **Incertidumbre** σ = desviación típica entre los 5 modelos, para cada componente de fuerza.
3. **Calibración**: σ se multiplica por un único factor *k*, ajustado en un conjunto de calibración para que la
   incertidumbre media coincida con el error medio: $k = \mathrm{RMSE}/\mathrm{RMV}$.
4. **Validación** en los demás conjuntos: tabla y tres figuras.

**Medidas** (fuerzas por componente, como en las tablas de MACE):

| medida | qué es | valor ideal |
|---|---|---|
| RMSE | $\sqrt{\langle (F_\mathrm{comité} - F_\mathrm{DFT})^2 \rangle}$, el error real | lo más bajo posible |
| RMV | $\sqrt{\langle (k\sigma)^2 \rangle}$, el error que predice el comité | igual al RMSE |
| RMSE/RMV | > 1: el comité subestima su error; < 1: lo sobrestima | 1 |
| ρ de Spearman | ¿ordena σ bien qué configuraciones tienen más error? | cerca de 1 |
| dentro de ±2σ | % de componentes con error menor que 2kσ | ≈ 95 % |

**Carpeta** (la del notebook): `models/FT_MACE_v32_c1.model … c5.model` y los `.xyz` con energía y fuerzas DFT.
Los resultados (tabla y figuras en PDF y PNG) se guardan en `resultados_uq/`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # evita que muera el kernel (OpenMP de numpy y torch)

import warnings
warnings.filterwarnings("ignore", message=".*weights_only.*")   # aviso de torch.load al abrir los .model (inofensivo con modelos propios)
warnings.filterwarnings("ignore", category=FutureWarning)       # avisos de funciones que cambiarán en versiones futuras de PyTorch

import glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from ase.io import read
from mace.calculators import MACECalculator
from tqdm import tqdm   # barra de progreso en texto (no necesita ipywidgets)

# ------------------------------------ Parámetros ------------------------------------
MODELOS = sorted(glob.glob("models/FT_MACE_v32_c*.model"))

# Conjuntos a evaluar:  nombre -> (archivo, frames). Los frames usan la sintaxis de ASE:
# ":" todos, "0:1250" los 1250 primeros, "1250:" del 1250 al final, "::5" uno de cada cinco.
# Los que no existan en la carpeta se saltan. Añade aquí los que quieras comparar.
CONJUNTOS = {
    "NVT 2250 K (0–2.4 ps)":   ("bgdy_2250K_nvt.xyz", "0:1250"),
    "NVT 2250 K (2.4–4.8 ps)": ("bgdy_2250K_nvt.xyz", "1250:"),
    "Test v32":                ("test_v32_filtrado_total.xyz", ":"),
}
CALIBRACION = "NVT 2250 K (0–2.4 ps)"   # conjunto con el que se ajusta k (los demás sirven para validar)

CLAVE_E, CLAVE_F = "energy_qe", "forces_qe"   # nombres de la energía y las fuerzas DFT en los xyz
DEVICE = "cuda"      # GPU; el notebook se detiene si PyTorch no la ve
DTYPE = "float64"    # el de entrenamiento; "float32" va mucho más rápido en GPU de consumo (GeForce)
SALIDA = "resultados_uq"
# ------------------------------------------------------------------------------------

os.makedirs(SALIDA, exist_ok=True)
print(f"{len(MODELOS)} modelos:", [os.path.basename(m) for m in MODELOS])

## 1. Evaluar el comité

`MACECalculator` con una lista de modelos evalúa los 5 a la vez y devuelve la predicción de cada uno
(`energy_comm`, `forces_comm`). Las predicciones se guardan en `resultados_uq/predicciones.pkl`: si vuelves a
ejecutar, solo se evalúan los conjuntos nuevos. **Si cambias los frames o el archivo de un conjunto, borra ese
archivo** (o cambia el nombre del conjunto).

In [ ]:
import torch
if DEVICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            f"PyTorch no ve ninguna GPU (torch {torch.__version__}, CUDA {torch.version.cuda}). "
            "Si CUDA sale None, este entorno tiene PyTorch solo para CPU: instala la versión con CUDA "
            "(https://pytorch.org/get-started/locally/) y reinicia el kernel.")
    print("GPU:", torch.cuda.get_device_name(0))

calc = MACECalculator(model_paths=MODELOS, device=DEVICE, default_dtype=DTYPE)
print(f"{len(calc.models)} modelos cargados en {next(calc.models[0].parameters()).device} ({DTYPE})")


def leer(archivo, frames):
    confs = read(archivo, frames)
    return [a for a in confs if a.info.get("config_type") != "TOTAL"]   # quita las copias de add_test_total.py


def evaluar(confs):
    """Para cada configuración guarda la referencia DFT y la predicción de cada modelo."""
    res = []
    for a in tqdm(confs):
        r = {
            "n": len(a),
            "t_fs": a.info.get("time_fs", np.nan),
            "E_dft": a.info[CLAVE_E] if CLAVE_E in a.info else a.get_potential_energy(),
            "F_dft": np.array(a.arrays[CLAVE_F] if CLAVE_F in a.arrays else a.get_forces()),
        }
        a = a.copy()
        a.calc = calc
        a.get_potential_energy()
        r["E_mod"] = np.array(calc.results["energy_comm"])   # (M,)      eV
        r["F_mod"] = np.array(calc.results["forces_comm"])   # (M, n, 3) eV/Å
        res.append(r)
    return res


archivo_pred = os.path.join(SALIDA, "predicciones.pkl")
pred = pickle.load(open(archivo_pred, "rb")) if os.path.exists(archivo_pred) else {}
for nombre, (archivo, frames) in CONJUNTOS.items():
    if nombre in pred:
        continue
    if not os.path.exists(archivo):
        print(f"Se salta '{nombre}': no encuentro {archivo}")
        continue
    print(f"Evaluando '{nombre}'")
    pred[nombre] = evaluar(leer(archivo, frames))
    pickle.dump(pred, open(archivo_pred, "wb"))

for nombre in CONJUNTOS:
    if nombre in pred:
        print(f"{nombre}: {len(pred[nombre])} configuraciones")

## 2. Incertidumbre, calibración y tabla

Para cada conjunto se juntan todas las componentes de fuerza: el error $\Delta F$ de la media del comité y la
desviación típica σ entre los 5 modelos. El factor *k* se calcula solo con el conjunto `CALIBRACION`.

In [ ]:
def analizar(r):
    """Error y sigma por componente de fuerza; RMSE y RMV de cada configuración; error de energía."""
    F_media = [x["F_mod"].mean(0) for x in r]
    return {
        "dF": np.concatenate([Fm - x["F_dft"] for Fm, x in zip(F_media, r)]).ravel(),   # eV/Å
        "sF": np.concatenate([x["F_mod"].std(0) for x in r]).ravel(),                   # eV/Å
        "rmse_conf": np.array([np.sqrt(np.mean((Fm - x["F_dft"]) ** 2)) for Fm, x in zip(F_media, r)]),
        "rmv_conf": np.array([np.sqrt(np.mean(x["F_mod"].var(0))) for x in r]),
        "dE": np.array([(x["E_mod"].mean() - x["E_dft"]) / x["n"] for x in r]),        # eV/átomo
        "sE": np.array([x["E_mod"].std() / x["n"] for x in r]),
        "t_fs": np.array([x["t_fs"] for x in r]),
    }


A = {nombre: analizar(pred[nombre]) for nombre in CONJUNTOS if nombre in pred}

cal = A[CALIBRACION]
k = np.sqrt(np.mean(cal["dF"] ** 2) / np.mean(cal["sF"] ** 2))
print(f"k = {k:.2f}: la discrepancia entre los modelos subestima el error real {k:.1f} veces.\n")

filas = []
for nombre, a in A.items():
    rmse = np.sqrt(np.mean(a["dF"] ** 2))
    rmv = k * np.sqrt(np.mean(a["sF"] ** 2))
    filas.append({
        "conjunto": nombre + ("  [calibración]" if nombre == CALIBRACION else ""),
        "n conf.": len(a["dE"]),
        "RMSE F (meV/Å)": 1e3 * rmse,
        "RMV F (meV/Å)": 1e3 * rmv,
        "RMSE/RMV": rmse / rmv,
        "ρ Spearman": spearmanr(a["rmv_conf"], a["rmse_conf"])[0] if len(a["dE"]) > 2 else np.nan,
        "dentro de ±2σ (%)": 100 * np.mean(np.abs(a["dF"]) < 2 * k * a["sF"]),
        "RMSE E (meV/át)": 1e3 * np.sqrt(np.mean(a["dE"] ** 2)),
        "sesgo E (meV/át)": 1e3 * np.mean(a["dE"]),
        "σ E entre modelos (meV/át)": 1e3 * np.sqrt(np.mean(a["sE"] ** 2)),
    })
tabla = pd.DataFrame(filas).set_index("conjunto")
tabla.to_csv(os.path.join(SALIDA, "tabla_uq.csv"))
tabla.round(2)

## 3. Figuras

Estilo común para las tres figuras (tamaño de columna de revista, fuentes de 8–9 pt). Se guardan en PDF
(vectorial, para el artículo) y en PNG.

In [ ]:
plt.rcParams.update({
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "legend.fontsize": 7.5,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 300, "savefig.bbox": "tight",
})
COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
assert len(A) <= len(COLORES), "Más de 6 conjuntos: amplía COLORES o junta conjuntos"
color = {nombre: COLORES[i] for i, nombre in enumerate(A)}


def guardar(fig, nombre):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(SALIDA, f"{nombre}.{ext}"))

### Figura 1. Diagrama de calibración

Las componentes de fuerza se ordenan por su σ y se agrupan en 10 intervalos con el mismo número de datos. En cada
intervalo se compara la incertidumbre predicha (RMV) con el error real (RMSE). Sobre la diagonal, el comité está
bien calibrado; por encima, subestima su error; por debajo, lo sobrestima (Levi et al., 2022).

In [ ]:
def curva_calibracion(a, n_intervalos=10):
    s, e = k * a["sF"], a["dF"]
    grupos = np.array_split(np.argsort(s), n_intervalos)
    rmv = np.array([np.sqrt(np.mean(s[g] ** 2)) for g in grupos])
    rmse = np.array([np.sqrt(np.mean(e[g] ** 2)) for g in grupos])
    return 1e3 * rmv, 1e3 * rmse


curvas = {nombre: curva_calibracion(a) for nombre, a in A.items()}
todo = np.concatenate([np.r_[x, y] for x, y in curvas.values()])
lim = [todo.min() / 1.3, todo.max() * 1.3]

fig, ax = plt.subplots(figsize=(3.4, 3.2))
ax.plot(lim, lim, color="0.6", lw=1, zorder=0, label="calibración perfecta")
for nombre, (x, y) in curvas.items():
    ax.plot(x, y, "o-", color=color[nombre], lw=1.5, ms=4, mec="white", mew=0.6, label=nombre)
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim,
       xlabel="incertidumbre predicha, RMV (meV/Å)", ylabel="error real, RMSE (meV/Å)")
ax.legend(frameon=False, loc="upper left")
guardar(fig, "fig1_calibracion")
plt.show()

### Figura 2. Error frente a incertidumbre, configuración a configuración

Cada punto es una configuración: su RMSE de fuerzas frente a la RMV que predice el comité. Si σ es útil, los
puntos siguen la diagonal; una nube redonda querría decir que σ no distingue las configuraciones buenas de las
malas.

In [ ]:
nombres = list(A)
ncol = min(3, len(nombres))
nfil = int(np.ceil(len(nombres) / ncol))
todo = 1e3 * np.concatenate([np.r_[k * a["rmv_conf"], a["rmse_conf"]] for a in A.values()])
lim = [todo.min() / 1.3, todo.max() * 1.3]

fig, axs = plt.subplots(nfil, ncol, figsize=(2.4 * ncol, 2.4 * nfil), sharex=True, sharey=True, squeeze=False)
for ax, nombre in zip(axs.flat, nombres):
    a = A[nombre]
    ax.plot(lim, lim, color="0.6", lw=1, zorder=0)
    ax.scatter(1e3 * k * a["rmv_conf"], 1e3 * a["rmse_conf"], s=5, alpha=0.5, lw=0, color=color[nombre])
    rho = tabla["ρ Spearman"].iloc[nombres.index(nombre)]
    ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, title=f"{nombre}\nρ = {rho:.2f}")
for ax in axs.flat[len(nombres):]:
    ax.set_visible(False)
for ax in axs[:, 0]:
    ax.set_ylabel("RMSE (meV/Å)")
for ax in axs[-1, :]:
    ax.set_xlabel("RMV predicha (meV/Å)")
fig.tight_layout()
guardar(fig, "fig2_error_vs_incertidumbre")
plt.show()

### Figura 3. A lo largo de la trayectoria

Solo para los conjuntos con tiempo (`time_fs`, las AIMD). Arriba: error real de cada frame (color) y la
incertidumbre calibrada del comité (negro). Si las dos curvas suben y bajan juntas, el comité avisa cuando el
modelo se equivoca. Abajo: error de energía por átomo.

In [ ]:
tray = [n for n, a in A.items() if np.isfinite(a["t_fs"]).all()]
if not tray:
    print("Ningún conjunto tiene time_fs")
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.2), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    for nombre in tray:
        a = A[nombre]
        t = a["t_fs"] / 1000
        ax1.plot(t, 1e3 * a["rmse_conf"], color=color[nombre], lw=0.8, label=f"error, {nombre}")
        ax1.plot(t, 1e3 * k * a["rmv_conf"], color="black", lw=0.6, alpha=0.7, label="_")
        ax2.plot(t, 1e3 * a["dE"], color=color[nombre], lw=0.8)
    ax1.plot([], [], color="black", lw=0.6, alpha=0.7, label="incertidumbre calibrada (k·RMV)")
    ax1.set(yscale="log", ylabel="RMSE de fuerzas\nen el frame (meV/Å)")
    ax1.legend(frameon=False, ncol=len(tray) + 1, loc="lower left", bbox_to_anchor=(0, 1.0))
    ax2.axhline(0, color="0.6", lw=0.8)
    ax2.set(xlabel="t (ps)", ylabel="error de E\n(meV/át)")
    fig.align_ylabels()
    guardar(fig, "fig3_trayectoria")
    plt.show()

## 4. Cómo leer los resultados

- **k** dice cuánto subestima la discrepancia entre los 5 modelos el error real. Si todos los modelos parten del
  mismo modelo fundacional (mace-mh-1), se parecen mucho entre sí y *k* sale bastante mayor que 1: es normal y
  es justo lo que corrige la calibración.
- **RMSE/RMV ≈ 1 en los conjuntos de validación** quiere decir que *k* sirve fuera de donde se ajustó. Si en un
  conjunto sale claramente > 1, el comité se fía demasiado ahí (por ejemplo, configuraciones que ningún modelo
  ha visto, o un error que comparten los 5 modelos).
- **Figura 1**: si la curva es más plana que la diagonal, el error tiene una parte que los 5 modelos comparten y
  que el comité no puede ver; σ sigue siendo útil para detectar las configuraciones peores, pero no como barra
  de error átomo a átomo.
- **Energía**: un *sesgo E* mucho mayor que la *σ E entre modelos* indica un error sistemático común a los 5
  modelos; la discrepancia entre modelos no lo detecta. Un desplazamiento constante no cambia las fuerzas ni la
  dinámica, pero sí las comparaciones de energía entre estados.

## 5. Texto para la sección de métodos (borrador en inglés)

> *Uncertainty quantification.* We trained a committee of M = 5 MACE models by fine-tuning the same foundation
> model with different random seeds. The committee prediction is the mean of the members, and the uncertainty of
> each force component is the standard deviation across members, σ. Because all members start from the same
> pretrained weights, σ underestimates the actual error. Following Levi et al., we therefore rescaled it with a single
> factor *k*, chosen on a calibration set so that the root mean variance (RMV) matches the root mean squared error
> (RMSE), k = RMSE/RMV. A related rescaling of committee spreads is used by Musil et al. and Imbalzano et al.
> Calibration was assessed on held-out data with reliability diagrams (RMSE vs. RMV in bins of equal population),
> the Spearman rank correlation between the per-configuration uncertainty and error, and the fraction of force
> components whose error lies within ±2kσ.

Referencias:
- D. Levi, L. Gispan, N. Giladi, E. Fetaya, *Evaluating and Calibrating Uncertainty Prediction in Regression
  Tasks*, Sensors **22**, 5540 (2022). https://doi.org/10.3390/s22155540
- F. Musil, M. J. Willatt, M. A. Langovoy, M. Ceriotti, *Fast and Accurate Uncertainty Estimation in Chemical
  Machine Learning*, J. Chem. Theory Comput. **15**, 906 (2019).
- G. Imbalzano et al., *Uncertainty estimation for molecular dynamics and sampling*, J. Chem. Phys. **154**,
  074102 (2021). https://doi.org/10.1063/5.0036522